# Ensemble starter — text feature bagging, bootstrap bagging and spatial corrections

**Target:** aim for **sub-7% off-market MdAPE** on full-strength chronological evaluation. Our current best observed full-history MdAPE is **7.4857%**; sub-7 would require roughly a 6.5% relative reduction. This is a stretch goal, not a prediction.

This notebook starts the ensemble registry and a small **optional** feature/row-bagged XGBoost experiment. It uses the 47 selected binary-BoW terms and the training-fitted ~1,000 candidate vocabulary, instead of limiting the ensemble to Tensor GAM and Matérn corrections.

See [ENSEMBLE_CANDIDATES.md](ENSEMBLE_CANDIDATES.md) for the full roadmap. Keep off-market and list-aware models separate, and do not compare pilot raw scores to the old full-history benchmark.

**Storage:** new fitted snapshots live in `saved_models/off_market/<name>/` or
`saved_models/list_aware/<name>/`, while old root-level snapshots remain
readable as legacy. A bagging run saves **one shared fitted feature transformer
plus each individually addressable tree member** using `save_model(..., track="off_market")`.
Saving is enabled for the opt-in pilot; nothing is saved until training is enabled.

**Design:** train chronological feature providers on ALL eligible historical sales; then select binary-BoW columns and optionally bootstrap XGBoost fitting rows. Historical comparable-sale features are **not** regenerated from bootstrapped sales. Validation properties are never used to choose the vocabulary or training rows.


In [1]:
from IPython.display import display
import pandas as pd
from models.ensemble_bagging import starter_specs, SELECTED_TERMS

ENSEMBLE_SEED = 42
ALL_BAG_SPECS = starter_specs(ENSEMBLE_SEED)
print("Selected reference BoW terms:", len(SELECTED_TERMS))
print("Candidate member configurations:", len(ALL_BAG_SPECS))
display(pd.DataFrame([s.record() for s in ALL_BAG_SPECS]))

MODEL_REGISTRY = pd.DataFrame([
    ("xgb_selected_47", "binary_bow", "Reference 47 selected terms"),
    ("xgb_full_binary", "binary_bow", "Training-fitted broad vocabulary"),
    ("xgb_selected_plus_random", "feature_bag", "Keep selected + random additions"),
    ("xgb_weighted_text_bags", "feature_bag", "4x selected-term sampling prior"),
    ("xgb_row_bootstraps", "row_bag", "Same features, bootstrapped fitting rows"),
    ("xgb_feature_row_joint", "joint_bag", "Weighted term sampling + bootstrap"),
    ("xgb_splines_geo", "spatial_features", "Tuned XGBoost with splines + geography"),
    ("xgb_tensor_gam", "residual", "GAM correction on true earlier OOT errors"),
    ("xgb_spatial_matern", "residual", "Tuned spatial GP correction"),
    ("xgb_spacetime_matern", "residual", "Tuned spatiotemporal GP correction"),
    ("lgb_catboost", "other_models", "Future independently tuned challenger"),
], columns=["model_id", "diversity_source", "purpose"])
display(MODEL_REGISTRY)


Selected reference BoW terms: 47
Candidate member configurations: 11


,name,text_mode,n_text,selected_weight,row_mode,row_fraction,seed
0,selected_47_reference,selected,150,4.0,all,1.0,42
1,full_vocab_binary,full,150,4.0,all,1.0,42
2,selected_plus_100,selected_plus,100,4.0,all,1.0,43
3,uniform_150_a,uniform,150,4.0,all,1.0,44
4,weighted_150_a,weighted,150,4.0,all,1.0,45
5,weighted_150_b,weighted,150,4.0,all,1.0,46
6,weighted_150_bootstrap_a,weighted,150,4.0,bootstrap,1.0,47
7,weighted_150_bootstrap_b,weighted,150,4.0,bootstrap,1.0,48
8,selected_47_bootstrap_a,selected,150,4.0,bootstrap,1.0,49
9,selected_47_bootstrap_b,selected,150,4.0,bootstrap,1.0,50


,model_id,diversity_source,purpose
0,xgb_selected_47,binary_bow,Reference 47 selected terms
1,xgb_full_binary,binary_bow,Training-fitted broad vocabulary
2,xgb_selected_plus_random,feature_bag,Keep selected + random additions
3,xgb_weighted_text_bags,feature_bag,4x selected-term sampling prior
4,xgb_row_bootstraps,row_bag,"Same features, bootstrapped fitting rows"
5,xgb_feature_row_joint,joint_bag,Weighted term sampling + bootstrap
6,xgb_splines_geo,spatial_features,Tuned XGBoost with splines + geography
7,xgb_tensor_gam,residual,GAM correction on true earlier OOT errors
8,xgb_spatial_matern,residual,Tuned spatial GP correction
9,xgb_spacetime_matern,residual,Tuned spatiotemporal GP correction


## 1 · Train a small text / row-bagging pilot (opt-in)

We start with August 2025 rather than the repeatedly analyzed Sep–Nov months. This is **development screening**, not an untouched final test.

The pilot keeps the existing tuned XGBoost parameter family, uses 250 trees and at most 6,000 historical sales, and compares 6 complementary variations. Use the same validation homes and dates for every model. If you add or remove members, **change the output directory** to avoid mixing cached results.

The first run may be slower while the full housing feature matrix and historical comparables are built; only the estimator fitting rows and binary text columns vary across bag members.


In [2]:
from ensemble_feature_bagging_pilot import run_feature_bagging_pilot

RUN_ENSEMBLE_BAG_PILOT = False  # Change to True to train the first candidates
SAVE_FITTED_ENSEMBLE_MODELS = True  # Save the full reusable bundle when training
BAG_PILOT_SPECS = (
    ALL_BAG_SPECS[0],  # 47 selected terms / all eligible training rows
    ALL_BAG_SPECS[1],  # full 1000-ish binary vocabulary
    ALL_BAG_SPECS[2],  # selected 47 plus random 100
    ALL_BAG_SPECS[3],  # uniform 150-term bag
    ALL_BAG_SPECS[4],  # weighted 150-term bag
    ALL_BAG_SPECS[6],  # weighted 150-term bag + row bootstrap
    ALL_BAG_SPECS[8],  # selected 47 + row bootstrap
)

BAG_PILOT_OUTPUT = "outputs/ensemble_feature_bagging_smoke"
if RUN_ENSEMBLE_BAG_PILOT:
    bag_scores, bag_predictions, bag_manifest = run_feature_bagging_pilot(
        validation_month="2025-08",
        train_start="2022-01-01",
        train_cap=6_000,
        validation_cap=350,
        trees=250,
        specs=BAG_PILOT_SPECS,
        seed=ENSEMBLE_SEED,
        max_extra_terms=1_000,
        min_df=10,
        output_dir=BAG_PILOT_OUTPUT,
        resume=True,
        save_fitted_models=SAVE_FITTED_ENSEMBLE_MODELS,
    )
    display(bag_scores.round(4))
    display(bag_manifest[[
        "name", "text_mode", "n_terms", "row_mode",
        "n_training_rows", "unique_training_rows"
    ]])
    print("Predictions and exact selected terms are saved in:", BAG_PILOT_OUTPUT)
    if SAVE_FITTED_ENSEMBLE_MODELS:
        print("Fitted ensemble bundle is stored under saved_models/off_market/")


In [3]:
# Simple PREDECLARED log-price blends; no tuning against the scoring homes.
# This is exploratory—don't promote based solely on the best August result.
from pathlib import Path
from models.ensemble_blending import score_blends
from models.ensemble_bagging import starter_specs

pilot_path = Path(BAG_PILOT_OUTPUT) / "predictions.csv"
if pilot_path.is_file():
    bag_predictions = pd.read_csv(pilot_path, dtype={"ml_num": str})
    fixed_blends = {
        "equal_reference_weighted": {
            "selected_47_reference": 0.5, "weighted_150_a": 0.5
        },
        "equal_reference_plus100": {
            "selected_47_reference": 0.5, "selected_plus_100": 0.5
        },
        "reference_weighted_bootstrap": {
            "selected_47_reference": 0.5,
            "weighted_150_bootstrap_a": 0.5
        },
        "equal_3_member_bag": {
            "selected_47_reference": 1,
            "selected_plus_100": 1,
            "weighted_150_bootstrap_a": 1,
        },
    }
    available = set(bag_predictions.member)
    fixed_blends = {
        k: v for k, v in fixed_blends.items() if set(v).issubset(available)
    }
    if fixed_blends:
        blend_scores, blend_predictions = score_blends(
            bag_predictions, fixed_blends
        )
        display(blend_scores.round(4))
        blend_predictions.to_csv(
            Path(BAG_PILOT_OUTPUT) / "fixed_blend_predictions.csv",
            index=False,
        )
        # Error correlation identifies duplication, not necessarily diversity gains.
        wide = bag_predictions.pivot(
            index=["month", "ml_num"], columns="member",
            values="predicted_price"
        )
        actual = bag_predictions.groupby(["month", "ml_num"]).sold_price.first()
        import numpy as np
        log_error = pd.DataFrame(
            {col: np.log(actual.loc[wide.index].to_numpy()) -
                  np.log(wide[col].to_numpy())
             for col in wide.columns},
            index=wide.index,
        )
        display(log_error.corr().round(3))
else:
    print("Run the opt-in bagging pilot first; no training occurs in this cell.")


Run the opt-in bagging pilot first; no training occurs in this cell.


## 2 · Separate diagnostic: are the spatial residual corrections complementary?

This block reads previously saved Matérn confirmation predictions and shows only **fixed 50/50 log-price blends** with Tensor GAM. No new GP/XGBoost fit is required.

**Important:** September–November 2025 have already been examined repeatedly, so this is an exploratory complementarity check, *not a legitimate untouched ensemble-selection result*. Do not tune blend weights on these predictions.


In [4]:
from pathlib import Path
from models.ensemble_blending import score_blends
from IPython.display import display

spatial_file = Path(
    "outputs/spacetime_matern_chronological_tuning/confirmation_predictions.csv"
)
if spatial_file.is_file():
    spatial_predictions = pd.read_csv(spatial_file, dtype={"ml_num": str})
    spatial_blends = {
        "gam_spatial_50_50": {
            "xgb_tensor_gam": .5, "xgb_spatial_matern_tuned": .5
        },
        "gam_spacetime_50_50": {
            "xgb_tensor_gam": .5, "xgb_spacetime_matern_tuned": .5
        },
        "gam_spatial_spacetime_equal": {
            "xgb_tensor_gam": 1,
            "xgb_spatial_matern_tuned": 1,
            "xgb_spacetime_matern_tuned": 1,
        },
    }
    spatial_scores, spatial_new_preds = score_blends(
        spatial_predictions, spatial_blends, variant_col="variant"
    )
    singles = spatial_predictions.groupby("variant").apply(
        lambda g: pd.Series({
            "homes": len(g),
            "mdape": (100*abs(g.predicted_price-g.sold_price)/g.sold_price).median(),
            "mape": (100*abs(g.predicted_price-g.sold_price)/g.sold_price).mean(),
        }),
        include_groups=False,
    ).reset_index().rename(columns={"variant": "member"})
    display(pd.concat([singles, spatial_scores], ignore_index=True)
            .sort_values(["mdape", "mape"]).round(4))
else:
    print("No local Matérn confirmation CSV yet; this section is optional.")


,member,homes,mdape,mape
4,gam_spacetime_50_50,2700.0,10.6762,13.4945
5,gam_spatial_spacetime_equal,2700.0,10.7030,13.4398
1,xgb_spacetime_matern_tuned,2700.0,10.7129,13.6247
3,xgb_tensor_gam,2700.0,10.7983,13.4371
0,baseline,2700.0,10.8761,13.5692
6,gam_spatial_50_50,2700.0,10.8780,13.3906
2,xgb_spatial_matern_tuned,2700.0,10.9641,13.3782


## 3 · Does blending add information, or only shrink corrections?

**Prediction-only, no retraining.** Read the Matérn chronological confirmation CSV, which contains the identical September–November homes under `baseline`, `xgb_tensor_gam` and both tuned Matérn corrections.

All models share the same XGBoost baseline. Recover each model's **effective** log correction as

\[
r_G=\log(P_{GAM})-\log(P_{XGB}),\qquad
r_M=\log(P_{Matérn})-\log(P_{XGB}).
\]

These are already **clipped and scaled** by their respective earlier fitted correction strength. This experiment does *not* refit GAM, GP, XGBoost or blending weights.

Compare:
- GAM and each Matérn alone;
- `0.5*r_G` and `0.5*r_M` (individual shrinkage controls);
- `0.5*r_G + 0.5*r_M` (the existing 50/50 geometric blend);
- `0.25*r_G + 0.25*r_M` (a doubly-shrunk blend control).

The important paired test is whether the blend beats **both individually shrunk** corrections on the *same homes*, in MdAPE and MAPE. Results include month-specific scores, month-stratified paired bootstrap intervals, per-home improvements, and correlation of the effective log corrections.

**Research caveat:** Sep–Nov 2025 has already been examined repeatedly; this is a retrospective diagnostic, *not independent evidence* to promote weights into the final ensemble. Do not pick or optimize weights from these scores.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from ensemble_shrinkage_diagnostic import run_shrinkage_vs_blending

RUN_SHRINKAGE_VS_BLEND = True  # Only reads the existing confirmation predictions
SHRINKAGE_SOURCE = Path(
    "outputs/spacetime_matern_chronological_tuning/confirmation_predictions.csv"
)
SHRINKAGE_OUTPUT = "outputs/ensemble_shrinkage_vs_spatial_blending"

if RUN_SHRINKAGE_VS_BLEND:
    if not SHRINKAGE_SOURCE.is_file():
        print(
            "Missing Matérn confirmation predictions:", SHRINKAGE_SOURCE,
            "\nFirst run the frozen Matérn confirmation experiment."
        )
    else:
        original_predictions = pd.read_csv(
            SHRINKAGE_SOURCE, dtype={"ml_num": str}
        )
        (shrink_pooled, shrink_monthly, shrink_pairs,
         shrink_correlations, shrink_predictions) = run_shrinkage_vs_blending(
            original_predictions,
            shrinkage=0.5,
            bootstrap_repeats=500,
            seed=42,
            output_dir=SHRINKAGE_OUTPUT,
        )
        print("All fixed correction strategies (lower is better):")
        display(shrink_pooled.round(4))
        print("Most revealing comparison: fixed 50/50 vs controls")
        display(shrink_pairs[[
            "challenger", "reference", "delta_mdape_pp",
            "mdape_ci_low", "mdape_ci_high",
            "delta_mape_pp", "mape_ci_low", "mape_ci_high",
            "homes_improved_pct", "months_better_mdape",
        ]].round(4))
        print("Monthly MdAPE / MAPE:")
        display(shrink_monthly[[
            "month", "variant", "homes", "mdape", "mape"
        ]].round(4))
        print("Effective correction-pattern correlations:")
        display(shrink_correlations.round(4))
        print("Saved predictions and comparisons:", SHRINKAGE_OUTPUT)

## Next checkpoint: iterative ensemble factory

The new **shrinkage-versus-blending** test helps determine whether
GAM + Matérn is genuinely complementary or simply a lower-strength GP/GAM
correction. Keep both its per-home predictions and paired comparisons in the
ensemble candidate registry. After that, the most valuable new signal is whether **weighted term bags + bootstrap samples** generate good predictions with meaningfully different errors from the 47-feature reference. Select promising members using *earlier out-of-time folds*; then promote a small, frozen ensemble to larger 700-tree/full-history training.

Keep model roster, training cutoffs, feature bag terms, bootstrap seeds, per-home predictions and fixed or OOT-fitted blend weights together. Without these, apparent ensemble gains are hard to reproduce. For a production-facing score below 7%, confirm on a genuinely separate chronological period if available; the old Dec 2025–May 2026 six-month benchmark has already been inspected.


## 3 · Load a fitted ensemble bundle (without retraining)

The model snapshot includes the already-fitted feature provider, its training-only text vocabulary, the selected per-member feature columns, fitted XGBoost estimators, and the log-target offset. Track-specific storage lets similarly named off-market and list-aware bundles coexist. The original dual-track `saved_models/XGBoost_text_binary` remains legacy and unchanged.

You can load the saved artifact with `load_model(name, track="off_market")`. Use `predict_member(hidden_homes, member_name)` for an individual ensemble member, `predict_members(hidden_homes)` to obtain all member predictions, or `predict(hidden_homes, weights=...)` for a fixed-weight geometric blend.


In [5]:
from save_model import list_saved_models, list_model_catalog, load_model
print("Saved off-market snapshots:", list_saved_models(track="off_market"))
print("Saved list-aware snapshots:", list_saved_models(track="list_aware"))
display(pd.DataFrame(list_model_catalog()))

# Enable only after the pilot has saved at least one fitted ensemble bundle.
LOAD_SAVED_ENSEMBLE = False
SAVED_ENSEMBLE_NAME = ""  # Use the name printed by the save step.
if LOAD_SAVED_ENSEMBLE:
    if not SAVED_ENSEMBLE_NAME:
        raise ValueError("Set SAVED_ENSEMBLE_NAME to an off-market bundle name")
    saved_ensemble = load_model(
        SAVED_ENSEMBLE_NAME, track="off_market"
    )
    print("Available fitted members:", sorted(saved_ensemble.members))
    print("Example: saved_ensemble.predict_member(hidden_homes, 'weighted_150_a')")


Saved off-market snapshots: []
Saved list-aware snapshots: []


,track,name,relative_path
0,legacy,CatBoost_Optuna_v1,CatBoost_Optuna_v1/
1,legacy,LightGBM,LightGBM/
2,legacy,LightGBM_Optuna_v1,LightGBM_Optuna_v1/
3,legacy,LightGBM_text_binary,LightGBM_text_binary/
4,legacy,XGBoost,XGBoost/
5,legacy,XGBoost_Optuna_v1,XGBoost_Optuna_v1/
6,legacy,XGBoost_text_binary,XGBoost_text_binary/
7,legacy,catboost_spatiotemporal_900,catboost_spatiotemporal_900/


---

## 4 · Iterative AutoML ensemble factory (first working implementation)

**Pipeline:** randomized 47-prioritized binary-BoW feature bags → independent bootstrap/subbag training rows → randomized legal space-time feature add-ons → cheap diagnostic screen → bounded **Optuna** tuning on early months → ensemble-weight admission on later matched homes → a **separately enabled, one-time** shadow-audit report after selection.

Only **XGBoost** is trainable in version 1; the ensemble-selection layer is separate so we can add CatBoost, LightGBM, and model-specific Tensor GAM / Matérn corrections once this foundation is validated.

Default cohorts:
- **March–April 2025:** Optuna training objective, preceding-month historical fitting.
- **May–June 2025:** candidate acceptance and fixed nonnegative log-price blend weights.
- **July–August 2025:** audit predictions, not consulted for parameter choice or acceptance.

These dates have been involved in earlier research. The last segment is shielded from **this factory's** decisions, **not a newly untouched competition holdout**. The repeatedly inspected December 2025–May 2026 six-month benchmark is also not a new holdout.

**Admission criterion:** a candidate must improve the current ensemble's `0.65 × MdAPE + 0.35 × MAPE` objective by at least **0.02 percentage points**, with MAPE deterioration no larger than **0.03 points**. Trial blends use a small, predefined weight grid. This still reuses the acceptance months across iterations, so keep the first run extremely small and treat the audit gap as a warning sign rather than a green light for unlimited tuning.

**Audit guard:** Leave `RUN_AUTOML_SHADOW_AUDIT=False` while developing.
After the roster is frozen, enable it once to inspect the audit. The
factory will then refuse to add new generations to that output directory,
preventing feedback from the revealed audit results.

**Stop and resume:** You can press **Interrupt Kernel / Ctrl+C**, close the notebook, or restart your machine. Re-running the SAME factory cell, with the same `output_dir` and settings, automatically restores the committed candidate roster, prior Optuna trials, previously completed tuning folds and cached held-out predictions. A model fit stopped in mid-fit is restarted; boosting rounds within that fit are **not** checkpointed. The factory does not continue training while the notebook is closed.

New files: `progress.json` reports the current stage; `roster.json` is the authoritative checkpoint, written atomically. A persistent `.run.lock` protects against accidentally running two factory processes in the same directory; its OS lock releases when the process dies, so **do not delete it manually**. You can safely increase `AUTOML_GENERATIONS` to extend the same protocol **until you open the shadow audit**. Other settings must remain fixed.

**Saving:** Optuna studies, candidate forecasts, member roster, current weights and shadow-audit predictions persist in `outputs/ensemble_automl_factory_v1/`. Optionally, accepted fitted model bundles (including transformers and individual tree models) are saved in `saved_models/off_market/`; list-aware ensembles will be trained separately.

**The eventual full ensemble factory** will add independent model families, candidate replacement, truly separate chronological outer tests and full-strength 700-tree refits. This run is an exploratory engineering pilot, **not** a full final ensemble optimizer.


In [ ]:
# v1 factory run: safe, small default budget; explicit opt-in for training.
# Optional dependency, install once in the notebook terminal: pip install optuna
from ensemble_automl_factory import run_automl_factory, run_automl_shadow_audit
from IPython.display import display

RUN_AUTOML_FACTORY = False
AUTOML_GENERATIONS = 1
AUTOML_CANDIDATES_PER_GENERATION = 2
AUTOML_OPTUNA_TRIALS = 2
AUTOML_SAVE_ACCEPTED_MODELS = True

AUTOML_SETTINGS = dict(
        tuning_months=("2025-03", "2025-04"),
        selection_months=("2025-05", "2025-06"),
        audit_months=("2025-07", "2025-08"),
        train_start="2022-01-01",
        train_cap=5_000,
        validation_cap=250,
        trees=220,
        max_extra_terms=1_000,
        min_df=10,
        candidates_per_generation=AUTOML_CANDIDATES_PER_GENERATION,
        generations=AUTOML_GENERATIONS,
        optuna_trials=AUTOML_OPTUNA_TRIALS,
        minimum_ensemble_gain_pp=0.02,
        max_mape_regression_pp=0.03,
        max_members=8,
        seed=42,
        screen_max_mdape=22.0,
        save_accepted_snapshots=AUTOML_SAVE_ACCEPTED_MODELS,
        output_dir="outputs/ensemble_automl_factory_v1",
)
if RUN_AUTOML_FACTORY:
    automl_summary, automl_events, automl_selection_predictions = run_automl_factory(
        **AUTOML_SETTINGS
    )
    print("Final selected members / effective log-price weights:")
    display(pd.Series(automl_summary["effective_member_weights"]))
    display(pd.DataFrame([
        {"cohort": "selection", **automl_summary["selection_metrics"]},
    ]).round(4))
    display(automl_events)
    print("Cumulative run artifacts: outputs/ensemble_automl_factory_v1/")
    print("Audit still sealed. Open it ONCE in the separate cell below.")


In [ ]:
# One-time audit. DO NOT enable while you are still adding generations.
# The factory locks the roster after opening this diagnostic.
RUN_AUTOML_SHADOW_AUDIT = False

if RUN_AUTOML_SHADOW_AUDIT:
    frozen_summary, shadow_audit_predictions = run_automl_shadow_audit(
        **AUTOML_SETTINGS
    )
    print("Selection versus shadow audit:")
    display(pd.DataFrame([
        {"cohort": "selection", **frozen_summary["selection_metrics"]},
        {"cohort": "shadow audit", **frozen_summary["shadow_audit_metrics"]},
        {"cohort": "audit first reference",
         **frozen_summary["audit_first_reference_metrics"]},
    ]).round(4))
    print("Audit sealed to frozen roster at:",
          "outputs/ensemble_automl_factory_v1/shadow_audit_lock.json")


In [ ]:
# Inspect a completed or interrupted study without retraining anything.
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

factory_dir = Path("outputs/ensemble_automl_factory_v1")
for filename in ("progress.json", "roster.json", "summary.json", "events.json"):
    path = factory_dir / filename
    if path.is_file():
        print(filename)
        payload = json.loads(path.read_text(encoding="utf-8"))
        if filename == "progress.json":
            display(pd.Series(payload, name="Last durable stage"))
        elif filename == "events.json":
            display(pd.DataFrame(payload))
        elif filename == "roster.json":
            print("Accepted members:", [
                m["candidate"]["name"] for m in payload["members"]
            ])
            print("Processed candidate count:", len(payload["processed"]))
        else:
            display(pd.Series(payload["effective_member_weights"]))
            print("Selection:", payload["selection_metrics"])
            if "shadow_audit_metrics" in payload:
                print("Shadow audit:", payload["shadow_audit_metrics"])
            else:
                print("Shadow audit has NOT been opened.")
